In [7]:
import kagglehub
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from skimage.feature import graycomatrix, graycoprops
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
import os
import random



print("\nDownloading MosMedData")
mosmed_path = kagglehub.dataset_download("mathurinache/mosmeddata-chest-ct-scans-with-covid19")
print(f"MosMedData path: {mosmed_path}")



def set_seed(seed=999):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


MosMedData path: /kaggle/input/mosmeddata-chest-ct-scans-with-covid19


In [8]:
def load_frozen_dataset(base_path='/kaggle/input/datasets/aaryaupi/ct3-dataset',
                        data_root='/kaggle/input/datasets/aaryaupi/ct3-dataset/output_data'):
    from sklearn.model_selection import train_test_split
    full_df = pd.read_csv(os.path.join(base_path, 'full_balanced.csv'))
    train_df, temp_df = train_test_split(full_df, test_size=0.2, random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
    
    for df in [train_df, val_df, test_df]:
        for col in ['ct_path', 'mu_path', 'mask_path']:
            if col in df.columns:
                # Fix duplicated directory names in paths
                df[col] = df[col].str.replace('/ct_processed/ct_processed/', '/ct_processed/')
                df[col] = df[col].str.replace('/mu_processed/mu_processed/', '/mu_processed/')
                df[col] = df[col].str.replace('/mask_processed/mask_processed/', '/mask_processed/')
                
                # Replace /kaggle/working/ with the correct data_root path
                df[col] = df[col].str.replace('/kaggle/working/', data_root + '/')
                
                # Also handle /kaggle/input/output-data/ if present
                df[col] = df[col].str.replace('/kaggle/input/output-data/', data_root + '/')
    
    return train_df, val_df, test_df

In [9]:

class CTDataset_ARSIVAE(Dataset):
    def __init__(self, csv_path=None, df=None, compute_on_fly=True, attr_mean=None, attr_std=None):
        if df is not None:
            self.df = df.reset_index(drop=True)
        elif csv_path is not None:
            self.df = pd.read_csv(csv_path)
        else:
            raise ValueError("Either csv_path or df must be provided")
        
        self.compute_on_fly = compute_on_fly
        self.has_precomputed = self._check_precomputed_features()
        self.attr_mean = attr_mean
        self.attr_std = attr_std
        if not self.has_precomputed and not compute_on_fly:
            raise ValueError("CSV missing precomputed features")
    
    def _check_precomputed_features(self):
        required = ['mean_HU', 'HU_std', 'HU_p10', 'HU_p25', 'HU_p50', 'HU_p75', 'HU_p90',
                   'mask_area_pixels', 'mask_fraction', 'grad_mean', 'grad_std', 
                   'contrast', 'homogeneity', 'entropy']
        return all(col in self.df.columns for col in required)
    
    def __len__(self):
        return len(self.df)
    
    def _compute_hu_features(self, ct, mask):
        lung_pixels = mask > 0.5
        hu_values = ct[lung_pixels]
        if len(hu_values) == 0:
            return {k: 0.0 for k in ['mean_HU', 'HU_std', 'HU_p10', 'HU_p25', 'HU_p50', 'HU_p75', 'HU_p90']}
        return {
            'mean_HU': float(np.mean(hu_values)),
            'HU_std': float(np.std(hu_values)),
            'HU_p10': float(np.percentile(hu_values, 10)),
            'HU_p25': float(np.percentile(hu_values, 25)),
            'HU_p50': float(np.percentile(hu_values, 50)),
            'HU_p75': float(np.percentile(hu_values, 75)),
            'HU_p90': float(np.percentile(hu_values, 90))
        }
    
    def _compute_shape_features(self, mask, image_size=512*512):
        mask_area = float(np.sum(mask > 0.5))
        return {
            'mask_area_pixels': mask_area,
            'mask_fraction': mask_area / image_size
        }
    
    def _compute_gradient_features(self, ct, mask):
        grad_y, grad_x = np.gradient(ct)
        grad_magnitude = np.sqrt(grad_x**2 + grad_y**2)
        lung_pixels = mask > 0.5
        grad_in_lung = grad_magnitude[lung_pixels]
        if len(grad_in_lung) == 0:
            return {'grad_mean': 0.0, 'grad_std': 0.0}
        return {
            'grad_mean': float(np.mean(grad_in_lung)),
            'grad_std': float(np.std(grad_in_lung))
        }
    
    def _compute_texture_features(self, ct, mask):
        lung_pixels = mask > 0.5
        if lung_pixels.sum() == 0:
            return {'contrast': 0.0, 'homogeneity': 1.0, 'entropy': 0.0}
        ct_masked = ct.copy()
        ct_masked[~lung_pixels] = ct_masked[lung_pixels].min()
        ct_min = ct_masked[lung_pixels].min()
        ct_max = ct_masked[lung_pixels].max()
        if ct_max == ct_min:
            return {'contrast': 0.0, 'homogeneity': 1.0, 'entropy': 0.0}
        ct_normalized = ((ct_masked - ct_min) / (ct_max - ct_min) * 255).astype(np.uint8)
        try:
            glcm = graycomatrix(ct_normalized, distances=[1], 
                               angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                               levels=256, symmetric=True, normed=True)
            contrast = graycoprops(glcm, 'contrast').mean()
            homogeneity = graycoprops(glcm, 'homogeneity').mean()
            glcm_norm = glcm / (glcm.sum() + 1e-10)
            entropy = -np.sum(glcm_norm * np.log2(glcm_norm + 1e-10))
            return {
                'contrast': float(contrast),
                'homogeneity': float(homogeneity),
                'entropy': float(entropy)
            }
        except:
            return {'contrast': 0.0, 'homogeneity': 1.0, 'entropy': 0.0}
    
    def _compute_all_physics(self, ct, mask):
        ct_hu = (ct+1) * 1400 - 1000
        hu_feat = self._compute_hu_features(ct_hu, mask)
        shape_feat = self._compute_shape_features(mask)
        grad_feat = self._compute_gradient_features(ct, mask)
        texture_feat = self._compute_texture_features(ct, mask)
        attributes = np.array([
            hu_feat['mean_HU'], hu_feat['HU_std'], hu_feat['HU_p10'], 
            hu_feat['HU_p25'], hu_feat['HU_p50'], hu_feat['HU_p75'], hu_feat['HU_p90'],
            shape_feat['mask_area_pixels'], shape_feat['mask_fraction'],
            grad_feat['grad_mean'], grad_feat['grad_std'],
            texture_feat['contrast'], texture_feat['homogeneity'], texture_feat['entropy']
        ], dtype=np.float32)
        return attributes
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ct = np.load(row['ct_path'])
        mask = np.load(row['mask_path'])
        
        if self.has_precomputed and not self.compute_on_fly:
            attributes = np.array([
                row['mean_HU'], row['HU_std'], row['HU_p10'], row['HU_p25'], 
                row['HU_p50'], row['HU_p75'], row['HU_p90'],
                row['mask_area_pixels'], row['mask_fraction'],
                row['grad_mean'], row['grad_std'],
                row['contrast'], row['homogeneity'], row['entropy']
            ], dtype=np.float32)
        else:
            attributes = self._compute_all_physics(ct, mask)
        
        if self.attr_mean is not None and self.attr_std is not None:
            attributes = (attributes - self.attr_mean) / (self.attr_std + 1e-8)
        
        return {
            'ct': torch.FloatTensor(ct).unsqueeze(0),
            'mask': torch.FloatTensor(mask).unsqueeze(0),
            'attributes': torch.FloatTensor(attributes),
            'label': torch.tensor(row['label'], dtype=torch.long),
            'id': row['id']
        }

In [10]:
class Encoder(nn.Module):
    def __init__(self,latent_dim=64):
        super().__init__()
        self.conv=nn.Sequential(nn.Conv2d(1,32,4,2,1),nn.BatchNorm2d(32),nn.LeakyReLU(0.2),nn.Conv2d(32,64,4,2,1),nn.BatchNorm2d(64),nn.LeakyReLU(0.2),nn.Conv2d(64,128,4,2,1),nn.BatchNorm2d(128),nn.LeakyReLU(0.2),nn.Conv2d(128,256,4,2,1),nn.BatchNorm2d(256),nn.LeakyReLU(0.2),nn.Conv2d(256,512,4,2,1),nn.BatchNorm2d(512),nn.LeakyReLU(0.2))
        self.fc_mu=nn.Linear(512*16*16,latent_dim)
        self.fc_logvar=nn.Linear(512*16*16,latent_dim)
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m,(nn.Conv2d,nn.Linear)):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='leaky_relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias,0)
        nn.init.xavier_normal_(self.fc_mu.weight,gain=0.01)
        nn.init.constant_(self.fc_mu.bias,0)
        nn.init.xavier_normal_(self.fc_logvar.weight,gain=0.01)
        nn.init.constant_(self.fc_logvar.bias,-5)
    def forward(self,x):
        h=self.conv(x).view(x.size(0),-1)
        mu=self.fc_mu(h)
        logvar=torch.clamp(self.fc_logvar(h),-10,2)
        return mu,logvar

class Decoder(nn.Module):
    def __init__(self,latent_dim=64):
        super().__init__()
        self.fc=nn.Linear(latent_dim,512*16*16)
        self.deconv=nn.Sequential(nn.ConvTranspose2d(512,256,4,2,1),nn.BatchNorm2d(256),nn.LeakyReLU(0.2),nn.ConvTranspose2d(256,128,4,2,1),nn.BatchNorm2d(128),nn.LeakyReLU(0.2),nn.ConvTranspose2d(128,64,4,2,1),nn.BatchNorm2d(64),nn.LeakyReLU(0.2),nn.ConvTranspose2d(64,32,4,2,1),nn.BatchNorm2d(32),nn.LeakyReLU(0.2),nn.ConvTranspose2d(32,1,4,2,1),nn.Tanh())
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m,(nn.ConvTranspose2d,nn.Linear)):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='leaky_relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias,0)
    def forward(self,z):
        h=self.fc(z).view(z.size(0),512,16,16)
        return self.deconv(h)

class AttributePredictor(nn.Module):
    def __init__(self,latent_dim=64,n_attributes=14):
        super().__init__()
        self.input_layer=nn.Linear(latent_dim,256)
        self.bn1=nn.BatchNorm1d(256)
        self.res1=nn.Linear(256,256)
        self.bn_res1=nn.BatchNorm1d(256)
        self.res2=nn.Linear(256,256)
        self.bn_res2=nn.BatchNorm1d(256)
        self.fc2=nn.Linear(256,128)
        self.bn2=nn.BatchNorm1d(128)
        self.fc3=nn.Linear(128,n_attributes)
        self.dropout=nn.Dropout(0.1)
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m,nn.Linear):
                nn.init.kaiming_normal_(m.weight,nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias,0)
    def forward(self,z):
        x=F.relu(self.bn1(self.input_layer(z)))
        identity=x
        x=F.relu(self.bn_res1(self.res1(x)))
        x=self.bn_res2(self.res2(x))
        x=F.relu(x+identity)
        x=self.dropout(F.relu(self.bn2(self.fc2(x))))
        return self.fc3(x)

class ARSIVAE(nn.Module):
    def __init__(self,latent_dim=64,n_attributes=14):
        super().__init__()
        self.encoder=Encoder(latent_dim)
        self.decoder=Decoder(latent_dim)
        self.attr_predictor=AttributePredictor(latent_dim,n_attributes)
    def reparameterize(self,mu,logvar):
        std=torch.exp(0.5*logvar).clamp(min=1e-4,max=10)
        eps=torch.randn_like(std)
        return mu+eps*std
    def forward(self,x):
        mu,logvar=self.encoder(x)
        z=self.reparameterize(mu,logvar)
        recon=self.decoder(z)
        attrs=self.attr_predictor(mu)
        return recon,mu,logvar,attrs

def loss_function(recon,x,mu,logvar,pred_attrs,true_attrs,beta,lambda_attr):
    recon_loss=F.mse_loss(recon,x,reduction='mean')
    kl_loss=torch.clamp(-0.5*torch.mean(1+logvar-mu.pow(2)-logvar.exp()),0,1e4)
    attr_loss=F.mse_loss(pred_attrs,true_attrs,reduction='mean')
    total=recon_loss+beta*kl_loss+lambda_attr*attr_loss
    return total,recon_loss,kl_loss,attr_loss

def get_improved_schedule(epoch,num_epochs=50):
    if epoch < 20:
        # Phase 1: Immediate but gentle KL penalty
        beta = 0.0001 + 0.0001 * (epoch / 20)  # Start at 0.0001, reach 0.0002
        lambda_attr = 1.5
        
    elif epoch < 40:
        # Phase 2: Gradual increase
        progress = (epoch - 20) / 20
        beta = 0.0002 + 0.0003 * progress  # 0.0002 → 0.0005
        lambda_attr = 1.5 + 1.5 * progress  # 1.5 → 3.0
        
    else:
        beta = 0.0005
        lambda_attr = 3.0
        
    return beta, lambda_attr

def plot_reconstructions_epoch(model,loader,device,epoch,save_dir='recon_epochs'):
    os.makedirs(save_dir,exist_ok=True)
    model.eval()
    batch=next(iter(loader))
    x=batch['ct'][:8].to(device)
    with torch.no_grad():
        recon,_,_,_=model(x)
    x=x.cpu().numpy()
    recon=recon.cpu().numpy()
    fig,axes=plt.subplots(2,8,figsize=(16,4))
    for i in range(8):
        axes[0,i].imshow(x[i,0],cmap='gray')
        axes[0,i].axis('off')
        if i==0:
            axes[0,i].set_title('Original')
        axes[1,i].imshow(recon[i,0],cmap='gray')
        axes[1,i].axis('off')
        if i==0:
            axes[1,i].set_title('Reconstructed')
    plt.suptitle(f'Epoch {epoch}')
    plt.tight_layout()
    plt.savefig(f'{save_dir}/recon_epoch_{epoch:03d}.png',dpi=150,bbox_inches='tight')
    plt.close()

def train_epoch(model,loader,optimizer,device,beta,lambda_attr):
    model.train()
    total_loss=recon_loss=kl_loss=attr_loss=0
    n_batches=0
    pbar=tqdm(loader,desc='Training')
    for batch in pbar:
        x=batch['ct'].to(device)
        attrs=batch['attributes'].to(device)
        if torch.isnan(x).any() or torch.isinf(x).any():
            continue
        optimizer.zero_grad()
        recon,mu,logvar,pred_attrs=model(x)
        if torch.isnan(recon).any() or torch.isinf(recon).any():
            continue
        loss,r,k,a=loss_function(recon,x,mu,logvar,pred_attrs,attrs,beta,lambda_attr)
        if torch.isnan(loss) or torch.isinf(loss):
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()
        total_loss+=loss.item()
        recon_loss+=r.item()
        kl_loss+=k.item()
        attr_loss+=a.item()
        n_batches+=1
        pbar.set_postfix({'loss':f'{loss.item():.3f}','recon':f'{r.item():.3f}','kl':f'{k.item():.3f}','attr':f'{a.item():.3f}'})
    if n_batches==0:
        return float('nan'),float('nan'),float('nan'),float('nan')
    return total_loss/n_batches,recon_loss/n_batches,kl_loss/n_batches,attr_loss/n_batches

def validate(model,loader,device,beta,lambda_attr):
    model.eval()
    total_loss=recon_loss=kl_loss=attr_loss=0
    with torch.no_grad():
        for batch in loader:
            x=batch['ct'].to(device)
            attrs=batch['attributes'].to(device)
            recon,mu,logvar,pred_attrs=model(x)
            loss,r,k,a=loss_function(recon,x,mu,logvar,pred_attrs,attrs,beta,lambda_attr)
            total_loss+=loss.item()
            recon_loss+=r.item()
            kl_loss+=k.item()
            attr_loss+=a.item()
    n=len(loader)
    return total_loss/n,recon_loss/n,kl_loss/n,attr_loss/n

def train_improved(model,train_loader,val_loader,device,epochs=50):
    enc_params=list(model.encoder.parameters())
    dec_params=list(model.decoder.parameters())
    attr_params=list(model.attr_predictor.parameters())
    optimizer=optim.AdamW([{'params':enc_params,'lr':1e-4},{'params':dec_params,'lr':1e-4},{'params':attr_params,'lr':5e-4}],weight_decay=1e-5)
    scheduler=optim.lr_scheduler.CosineAnnealingLR(optimizer,epochs)
    history={'train_total':[],'val_total':[],'train_recon':[],'val_recon':[],'train_kl':[],'val_kl':[],'train_attr':[],'val_attr':[],'beta':[],'lambda':[]}
    best_val_attr_loss=float('inf')
    best_epoch=0
    for epoch in range(epochs):
        beta,lambda_attr=get_improved_schedule(epoch,epochs)
        history['beta'].append(beta)
        history['lambda'].append(lambda_attr)
        train_loss,train_r,train_k,train_a=train_epoch(model,train_loader,optimizer,device,beta,lambda_attr)
        val_loss,val_r,val_k,val_a=validate(model,val_loader,device,beta,lambda_attr)
        scheduler.step()
        history['train_total'].append(train_loss)
        history['val_total'].append(val_loss)
        history['train_recon'].append(train_r)
        history['val_recon'].append(val_r)
        history['train_kl'].append(train_k)
        history['val_kl'].append(val_k)
        history['train_attr'].append(train_a)
        history['val_attr'].append(val_a)
        phase="Physics" if epoch<15 else "Balance" if epoch<35 else "Finetune"
        print(f"Epoch {epoch+1}/{epochs} [{phase}] beta={beta:.4f} lambda={lambda_attr:.2f}")
        print(f"Train: Total={train_loss:.4f} Recon={train_r:.4f} KL={train_k:.4f} Attr={train_a:.4f}")
        print(f"Val: Total={val_loss:.4f} Recon={val_r:.4f} KL={val_k:.4f} Attr={val_a:.4f}")
        if(epoch+1)%5==0:
            plot_reconstructions_epoch(model,val_loader,device,epoch+1)
            print(f"Saved reconstruction for epoch {epoch+1}")
        if val_a<best_val_attr_loss:
            best_val_attr_loss=val_a
            best_epoch=epoch+1
            torch.save(model.state_dict(),'best_arsivae_improved.pth')
            print(f"Best model saved val_attr_loss={val_a:.4f}")
    print(f"Best model from epoch {best_epoch} with val_attr_loss={best_val_attr_loss:.4f}")
    return model,history

def extract_features(model,loader,device):
    model.eval()
    latents,labels,pred_attrs,true_attrs=[],[],[],[]
    with torch.no_grad():
        for batch in loader:
            x=batch['ct'].to(device)
            mu,_=model.encoder(x)
            attrs=model.attr_predictor(mu)
            latents.append(mu.cpu().numpy())
            labels.append(batch['label'].cpu().numpy())
            pred_attrs.append(attrs.cpu().numpy())
            true_attrs.append(batch['attributes'].cpu().numpy())
    return {'latents':np.vstack(latents),'labels':np.concatenate(labels),'pred_attrs':np.vstack(pred_attrs),'true_attrs':np.vstack(true_attrs)}

def plot_training_curves(history,save_path='training_curves.png'):
    fig,axes=plt.subplots(2,3,figsize=(15,8))
    epochs=range(1,len(history['train_total'])+1)
    axes[0,0].plot(epochs,history['train_total'],'b-',label='Train')
    axes[0,0].plot(epochs,history['val_total'],'r-',label='Val')
    axes[0,0].set_title('Total Loss')
    axes[0,0].legend()
    axes[0,0].grid(alpha=0.3)
    axes[0,1].plot(epochs,history['train_recon'],'b-',label='Train')
    axes[0,1].plot(epochs,history['val_recon'],'r-',label='Val')
    axes[0,1].set_title('Reconstruction Loss')
    axes[0,1].legend()
    axes[0,1].grid(alpha=0.3)
    axes[0,2].plot(epochs,history['train_kl'],'b-',label='Train')
    axes[0,2].plot(epochs,history['val_kl'],'r-',label='Val')
    axes[0,2].set_title('KL Divergence')
    axes[0,2].legend()
    axes[0,2].grid(alpha=0.3)
    axes[1,0].plot(epochs,history['train_attr'],'b-',label='Train')
    axes[1,0].plot(epochs,history['val_attr'],'r-',label='Val')
    axes[1,0].set_title('Attribute Loss')
    axes[1,0].legend()
    axes[1,0].grid(alpha=0.3)
    axes[1,1].plot(epochs,history['beta'],'purple')
    axes[1,1].set_title('Beta Schedule')
    axes[1,1].grid(alpha=0.3)
    axes[1,2].plot(epochs,history['lambda'],'orange')
    axes[1,2].set_title('Lambda Schedule')
    axes[1,2].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path,dpi=300,bbox_inches='tight')
    plt.close()

def plot_physics_alignment(data,save_path='physics_alignment.png'):
    pred=data['pred_attrs']
    true=data['true_attrs']
    attr_names=['mean_HU','HU_std','HU_p10','HU_p25','HU_p50','HU_p75','HU_p90','mask_area','mask_frac','grad_mean','grad_std','contrast','homog','entropy']
    fig,axes=plt.subplots(3,5,figsize=(20,12))
    axes=axes.flatten()
    r2_scores=[]
    for i in range(14):
        ax=axes[i]
        ax.scatter(true[:,i],pred[:,i],alpha=0.3,s=10,color='steelblue')
        min_val=min(true[:,i].min(),pred[:,i].min())
        max_val=max(true[:,i].max(),pred[:,i].max())
        ax.plot([min_val,max_val],[min_val,max_val],'r--')
        r2=r2_score(true[:,i],pred[:,i])
        r2_scores.append(r2)
        ax.set_xlabel(f'True {attr_names[i]}')
        ax.set_ylabel(f'Pred {attr_names[i]}')
        ax.set_title(f'{attr_names[i]} R2={r2:.3f}')
        ax.grid(alpha=0.3)
    axes[14].axis('off')
    plt.tight_layout()
    plt.savefig(save_path,dpi=300,bbox_inches='tight')
    plt.close()
    return r2_scores,np.mean(r2_scores)

def plot_latent_space(data,save_path='latent_space.png'):
    latents=data['latents']
    labels=data['labels']
    pred_attrs=data['pred_attrs']
    pca=PCA(n_components=2)
    latent_pca=pca.fit_transform(latents)
    fig,axes=plt.subplots(2,3,figsize=(18,12))
    ax=axes[0,0]
    colors=['#3498db','#e74c3c']
    for i,label_name in enumerate(['Normal','COVID']):
        mask=labels==i
        ax.scatter(latent_pca[mask,0],latent_pca[mask,1],c=colors[i],label=label_name,alpha=0.6,s=30)
    ax.set_title('Class Separation')
    ax.legend()
    ax.grid(alpha=0.3)
    physics_features=[('mean_HU',0,'Mean HU'),('grad_mean',9,'Gradient Mean'),('entropy',13,'Entropy'),('mask_area',7,'Mask Area'),('contrast',11,'Contrast')]
    for idx,(name,attr_idx,title) in enumerate(physics_features):
        ax=axes.flatten()[idx+1]
        scatter=ax.scatter(latent_pca[:,0],latent_pca[:,1],c=pred_attrs[:,attr_idx],cmap='viridis',alpha=0.6,s=30)
        ax.set_title(f'{title}')
        plt.colorbar(scatter,ax=ax,label=name)
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path,dpi=300,bbox_inches='tight')
    plt.close()

def compute_normalization_stats(dataset):
    all_attrs=[]
    for i in tqdm(range(len(dataset)),desc="Computing stats"):
        sample=dataset[i]
        all_attrs.append(sample['attributes'].numpy())
    all_attrs=np.vstack(all_attrs)
    mean=all_attrs.mean(axis=0)
    std=all_attrs.std(axis=0)
    std=np.where(std<1e-6,1.0,std)
    return mean,std

In [11]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import r2_score
import json


def compute_r2_scores(pred_attrs, true_attrs):
    """Compute per-feature R² scores"""
    attr_names = ['mean_HU', 'HU_std', 'HU_p10', 'HU_p25', 'HU_p50', 'HU_p75', 'HU_p90',
                  'mask_area', 'mask_frac', 'grad_mean', 'grad_std', 
                  'contrast', 'homog', 'entropy']
    
    r2_scores = []
    for i in range(14):
        r2 = r2_score(true_attrs[:, i], pred_attrs[:, i])
        r2_scores.append(r2)
    
    results_df = pd.DataFrame({
        'Feature': attr_names,
        'R²': r2_scores
    })
    
    return results_df, np.mean(r2_scores)


def diagnose_features(pred_attrs, true_attrs, attr_mean, attr_std):
    """Comprehensive feature diagnostics"""
    
    attr_names = ['mean_HU', 'HU_std', 'HU_p10', 'HU_p25', 'HU_p50', 'HU_p75', 'HU_p90',
                  'mask_area', 'mask_frac', 'grad_mean', 'grad_std', 
                  'contrast', 'homog', 'entropy']
    
    # 1. Check normalization stats
    print("\n[1] Normalization Statistics:")
    print(f"{'Feature':<15} {'Mean':>10} {'Std':>10} {'Status':>15}")
    for i, name in enumerate(attr_names):
        status = "✓ OK" if attr_std[i] > 0.01 else "⚠️  LOW STD"
        print(f"{name:<15} {attr_mean[i]:>10.4f} {attr_std[i]:>10.4f} {status:>15}")
    
    # 2. Check for constant predictions
    print("\n[2] Variance Check (Detecting Constant Predictions):")
    print(f"{'Feature':<15} {'Pred Std':>12} {'True Std':>12} {'Status':>15}")
    
    constant_features = []
    for i, name in enumerate(attr_names):
        pred_std = pred_attrs[:, i].std()
        true_std = true_attrs[:, i].std()
        
        if pred_std < 1e-6:
            status = "CONSTANT"
            constant_features.append(name)
        elif pred_std < 0.01:
            status = "LOW VAR"
        else:
            status = "OK"
        
        print(f"{name:<15} {pred_std:>12.6f} {true_std:>12.6f} {status:>15}")
    
    if constant_features:
        print(f"\nWARNING: {len(constant_features)} features have constant predictions!")
        for feat in constant_features:
            idx = attr_names.index(feat)
            print(f"   - {feat}: All predictions ≈ {pred_attrs[:, idx].mean():.6f}")
    
    # 3. Compute R² with detailed breakdown
    print("\n[3] R² Score Analysis:")
    
    print(f"{'Feature':<15} {'R²':>10} {'Pearson r':>12} {'Status':>15}")
    
    
    r2_scores = []
    for i, name in enumerate(attr_names):
        pred_i = pred_attrs[:, i]
        true_i = true_attrs[:, i]
        
        try:
            r2 = r2_score(true_i, pred_i)
            # Pearson correlation
            corr = np.corrcoef(pred_i, true_i)[0, 1]
        except:
            r2 = 0.0
            corr = 0.0
        
        r2_scores.append(r2)
        
        if r2 < 0.01:
            status = "FAILED"
        elif r2 < 0.5:
            status = "POOR"
        elif r2 < 0.8:
            status = "GOOD"
        else:
            status = "EXCELLENT"
        
        print(f"{name:<15} {r2:>10.4f} {corr:>12.4f} {status:>15}")
    
    avg_r2 = np.mean(r2_scores)
    print(f"{'MEAN R²':<15} {avg_r2:>10.4f}")
    
    # 4. Check prediction ranges
    print("\n[4] Prediction vs True Value Ranges:")
    print(f"{'Feature':<15} {'Pred Range':>25} {'True Range':>25}")
    
    for i, name in enumerate(attr_names):
        pred_range = f"[{pred_attrs[:, i].min():.4f}, {pred_attrs[:, i].max():.4f}]"
        true_range = f"[{true_attrs[:, i].min():.4f}, {true_attrs[:, i].max():.4f}]"
        print(f"{name:<15} {pred_range:>25} {true_range:>25}")
    return r2_scores, avg_r2

In [12]:
def main():
    SEED = 999
    set_seed(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"ARSIVAE TRAINING - SEED {SEED}")
    
    # Load frozen datasets with corrected paths
    print("Loading frozen datasets...")
    train_df, val_df, test_df = load_frozen_dataset(
        base_path='/kaggle/input/datasets/aaryaupi/ct3-dataset',
        data_root='/kaggle/input/datasets/aaryaupi/ct3-dataset'
    )
    print("\nFirst 3 CT paths in train_df:")
    print(train_df['ct_path'].head(3).tolist())
    print("\nFirst 3 mask paths in train_df:")
    print(train_df['mask_path'].head(3).tolist())
    
    first_ct_path = train_df['ct_path'].iloc[0]
    print(f"\nChecking if first CT file exists: {first_ct_path}")
    print(f"File exists: {os.path.exists(first_ct_path)}")
    
    print("\nComputing normalization statistics...")
    train_dataset_unnorm = CTDataset_ARSIVAE(df=train_df, compute_on_fly=True)
    attr_mean, attr_std = compute_normalization_stats(train_dataset_unnorm)

    # Needed when 3 features where collapsing towards default vale hence added this helper function to help help us diagnoise any normalization issues 
    print("\n" + "="*70)
    print("CHECKPOINT 1: Normalization Statistics")
    print("="*70)
    attr_names = ['mean_HU', 'HU_std', 'HU_p10', 'HU_p25', 'HU_p50', 'HU_p75', 'HU_p90',
                  'mask_area', 'mask_frac', 'grad_mean', 'grad_std', 
                  'contrast', 'homog', 'entropy']
    
    print(f"\n{'Feature':<15} {'Mean':>12} {'Std':>12} {'Check':>10}")
    print("-"*70)
    suspicious_features = []
    for i, name in enumerate(attr_names):
        check = "✓" if attr_std[i] > 0.01 else "⚠️ "
        if attr_std[i] < 0.01:
            suspicious_features.append((name, attr_std[i]))
        print(f"{name:<15} {attr_mean[i]:>12.4f} {attr_std[i]:>12.4f} {check:>10}")
    
    if suspicious_features:
        print(f"\nWARNING: {len(suspicious_features)} features have very low std (<0.01):")
        for feat, std_val in suspicious_features:
            print(f"   - {feat}: std = {std_val:.6f}")
        print("   These features may produce near-constant predictions (R² ≈ 0)")
    
    # Create normalized datasets
    print("\nCreating normalized datasets...")
    train_dataset = CTDataset_ARSIVAE(df=train_df, compute_on_fly=True, 
                                      attr_mean=attr_mean, attr_std=attr_std)
    val_dataset = CTDataset_ARSIVAE(df=val_df, compute_on_fly=True, 
                                    attr_mean=attr_mean, attr_std=attr_std)
    test_dataset = CTDataset_ARSIVAE(df=test_df, compute_on_fly=True, 
                                     attr_mean=attr_mean, attr_std=attr_std)
    
    print(f"Dataset sizes - Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, 
                             num_workers=4, pin_memory=True,
                             worker_init_fn=lambda worker_id: np.random.seed(SEED + worker_id))
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, 
                           num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, 
                            num_workers=4, pin_memory=True)
    
    
    print("\nInitializing model...")
    model = ARSIVAE(latent_dim=85, n_attributes=14).to(device)
    
    
    NUM_EPOCHS = 50
    print(f"\nStarting training for {NUM_EPOCHS} epochs...")
    model, history = train_improved(model, train_loader, val_loader, device, epochs=NUM_EPOCHS)
    
    
    print("CHECKPOINT 2: Training Summary")
    print(f"Final Train Loss: {history['train_total'][-1]:.4f}")
    print(f"Final Val Loss:   {history['val_total'][-1]:.4f}")
    print(f"Final Train KL:   {history['train_kl'][-1]:.4f}")
    print(f"Final Val KL:     {history['val_kl'][-1]:.4f}")
    print(f"Final Train Attr: {history['train_attr'][-1]:.4f}")
    print(f"Final Val Attr:   {history['val_attr'][-1]:.4f}")
    
    if history['val_kl'][-1] < 5.0:
        print("ARNING: Final KL < 5.0 - possible posterior collapse!")
    else:
        print("KL divergence looks healthy")
    
    # Load best model
    model.load_state_dict(torch.load('best_arsivae_improved.pth'))
    print("\n Loaded best model")
    
    # Plot training curves
    plot_training_curves(history, 'training_curves.png')
    
    # Extract features
    print("\n Extracting features...")
    train_data = extract_features(model, train_loader, device)
    val_data = extract_features(model, val_loader, device)
    test_data = extract_features(model, test_loader, device)
    
    print("CHECKPOINT 3: Feature Extraction Check")
    print(f"\nExtracted shapes:")
    print(f"  Val latents:    {val_data['latents'].shape}")
    print(f"  Val pred_attrs: {val_data['pred_attrs'].shape}")
    print(f"  Val true_attrs: {val_data['true_attrs'].shape}")
    
    print(f"\nPredicted attributes statistics:")
    print(f"  Min:  {val_data['pred_attrs'].min():.4f}")
    print(f"  Max:  {val_data['pred_attrs'].max():.4f}")
    print(f"  Mean: {val_data['pred_attrs'].mean():.4f}")
    print(f"  Std:  {val_data['pred_attrs'].std():.4f}")
    
    print(f"\nTrue attributes statistics:")
    print(f"  Min:  {val_data['true_attrs'].min():.4f}")
    print(f"  Max:  {val_data['true_attrs'].max():.4f}")
    print(f"  Mean: {val_data['true_attrs'].mean():.4f}")
    print(f"  Std:  {val_data['true_attrs'].std():.4f}")
    
    # Check if both are in normalized space (both should be close to 0 mean, ~1 std)
    pred_normalized = abs(val_data['pred_attrs'].mean()) < 1.0
    true_normalized = abs(val_data['true_attrs'].mean()) < 1.0
    
    if pred_normalized and true_normalized:
        print("\nBoth predictions and true values appear normalized")
    elif not pred_normalized and not true_normalized:
        print("\nNeither predictions nor true values are normalized")
    else:
        print("\nWARNING: Normalization mismatch detected!")
        print(f"   Predictions normalized: {pred_normalized}")
        print(f"   True values normalized: {true_normalized}")
    
    # ===== DIAGNOSTIC CHECKPOINT 4: Comprehensive Feature Analysis =====
    r2_scores, avg_r2 = diagnose_features(val_data['pred_attrs'], val_data['true_attrs'], 
                                          attr_mean, attr_std)
    
    print("TEST SET R² SCORES")
    
    test_r2_df, test_avg_r2 = compute_r2_scores(test_data['pred_attrs'], test_data['true_attrs'])
    print(test_r2_df.to_string(index=False))
    print(f"\nTest Avg R²: {test_avg_r2:.4f}")
    
    
    print("\nGenerating visualizations...")
    plot_physics_alignment(val_data, 'val_physics_alignment.png')
    plot_latent_space(val_data, 'val_latent_space.png')
    
    
    print("\nSaving outputs...")
    np.save('train_latents.npy', train_data['latents'])
    np.save('val_latents.npy', val_data['latents'])
    np.save('test_latents.npy', test_data['latents'])
    np.save('train_labels.npy', train_data['labels'])
    np.save('val_labels.npy', val_data['labels'])
    np.save('test_labels.npy', test_data['labels'])
    np.save('attr_mean.npy', attr_mean)
    np.save('attr_std.npy', attr_std)
    
    results = {
        'seed': SEED,
        'val_avg_r2': float(avg_r2),
        'test_avg_r2': float(test_avg_r2),
        'val_r2_scores': [float(r) for r in r2_scores],
        'test_r2_scores': test_r2_df['R²'].tolist(),
        'final_train_loss': float(history['train_total'][-1]),
        'final_val_loss': float(history['val_total'][-1]),
        'final_train_kl': float(history['train_kl'][-1]),
        'final_val_kl': float(history['val_kl'][-1]),
        'final_train_attr': float(history['train_attr'][-1]),
        'final_val_attr': float(history['val_attr'][-1]),
        'attr_names': attr_names,
        'attr_mean': attr_mean.tolist(),
        'attr_std': attr_std.tolist(),
        'suspicious_features': suspicious_features
    }
    
    with open(f'results_seed_{SEED}.json', 'w') as f:
        json.dump(results, f, indent=2)
    
    print("TRAINING COMPLETE - FINAL SUMMARY")
    print(f"Seed:              {SEED}")
    print(f"Val Avg R²:        {avg_r2:.4f}")
    print(f"Test Avg R²:       {test_avg_r2:.4f}")
    print(f"Final Val KL:      {history['val_kl'][-1]:.4f}")
    print(f"Final Val Attr:    {history['val_attr'][-1]:.4f}")
    
    return model, history, val_data


if __name__ == '__main__':
    model, history, val_data = main()
ory, val_data = main()

ARSIVAE TRAINING - SEED 999
Loading frozen datasets...

First 3 CT paths in train_df:
['/kaggle/input/datasets/aaryaupi/ct3-dataset/output_data/mosmed_normal_0013_slice_025_ct.npy', '/kaggle/input/datasets/aaryaupi/ct3-dataset/output_data/mosmed_covid_0024_slice_018_ct.npy', '/kaggle/input/datasets/aaryaupi/ct3-dataset/output_data/mosmed_covid_0031_slice_016_ct.npy']

First 3 mask paths in train_df:
['/kaggle/input/datasets/aaryaupi/ct3-dataset/output_data/mosmed_normal_0013_slice_025_mask.npy', '/kaggle/input/datasets/aaryaupi/ct3-dataset/output_data/mosmed_covid_0024_slice_018_mask.npy', '/kaggle/input/datasets/aaryaupi/ct3-dataset/output_data/mosmed_covid_0031_slice_016_mask.npy']

Checking if first CT file exists: /kaggle/input/datasets/aaryaupi/ct3-dataset/output_data/mosmed_normal_0013_slice_025_ct.npy
File exists: True

Computing normalization statistics...


Computing stats: 100%|██████████| 1456/1456 [01:35<00:00, 15.19it/s]



CHECKPOINT 1: Normalization Statistics

Feature                 Mean          Std      Check
----------------------------------------------------------------------
mean_HU             332.0749     162.3619          ✓
HU_std              773.4240      81.0512          ✓
HU_p10             -754.9590     132.8568          ✓
HU_p25             -469.8891     461.9054          ✓
HU_p50              654.1099     304.4975          ✓
HU_p75              996.9705      57.9168          ✓
HU_p90             1088.8024      34.9888          ✓
mask_area        144964.5156   25731.3984          ✓
mask_frac             0.5530       0.0982          ✓
grad_mean             0.0748       0.0220          ✓
grad_std              0.1190       0.0161          ✓
contrast            325.3487     138.9616          ✓
homog                 0.5774       0.0807          ✓
entropy               9.2432       1.1401          ✓

Creating normalized datasets...
Dataset sizes - Train: 1456, Val: 182, Test: 182

Initializi

Training: 100%|██████████| 46/46 [00:23<00:00,  1.97it/s, loss=0.773, recon=0.251, kl=13.683, attr=0.347]


Epoch 1/50 [Physics] beta=0.0001 lambda=1.50
Train: Total=1.5481 Recon=0.4282 KL=10.3123 Attr=0.7459
Val: Total=1.1250 Recon=0.2378 KL=9.1173 Attr=0.5909
Best model saved val_attr_loss=0.5909


...

Training: 100%|██████████| 46/46 [00:22<00:00,  2.05it/s, loss=0.255, recon=0.022, kl=17.602, attr=0.075]


Epoch 49/50 [Finetune] beta=0.0005 lambda=3.00
Train: Total=0.2263 Recon=0.0201 KL=17.5749 Attr=0.0658
Val: Total=0.1667 Recon=0.0344 KL=17.6316 Attr=0.0411


Training: 100%|██████████| 46/46 [00:22<00:00,  2.05it/s, loss=0.202, recon=0.020, kl=16.684, attr=0.058]


Epoch 50/50 [Finetune] beta=0.0005 lambda=3.00
Train: Total=0.2338 Recon=0.0202 KL=17.5555 Attr=0.0683
Val: Total=0.1691 Recon=0.0344 KL=17.5638 Attr=0.0420
Saved reconstruction for epoch 50
Best model from epoch 44 with val_attr_loss=0.0379
CHECKPOINT 2: Training Summary
Final Train Loss: 0.2338
Final Val Loss:   0.1691
Final Train KL:   17.5555
Final Val KL:     17.5638
Final Train Attr: 0.0683
Final Val Attr:   0.0420
KL divergence looks healthy

 Loaded best model

 Extracting features...
CHECKPOINT 3: Feature Extraction Check

Extracted shapes:
  Val latents:    (182, 85)
  Val pred_attrs: (182, 14)
  Val true_attrs: (182, 14)

Predicted attributes statistics:
  Min:  -4.1057
  Max:  9.1739
  Mean: 0.0057
  Std:  1.0112

True attributes statistics:
  Min:  -4.2763
  Max:  10.5049
  Mean: 0.0208
  Std:  1.0869

Both predictions and true values appear normalized

[1] Normalization Statistics:
Feature               Mean        Std          Status
mean_HU           332.0749   162.3619

ValueError: too many values to unpack (expected 2)